# Silver Exploration Notebook


## Imports

In [ ]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    sum as spark_sum
)

## Start Spark

In [4]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/02 11:40:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

/home/ubuntu/spark-lab-data/dev/project/raw


## Read Bike-sharing dataset
+ Adding city and month

In [ ]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn(
        "year_month",
        element_at(split(col("src"), "/"), -1)
    )
    .withColumn(
        "year_month",
        split(col("year_month"), r"\.").getItem(0)
    )
)

trips_raw.select("city", "year_month").show(5)

## Exporation of Bike-sharing dataset

In [ ]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------------------+--------------------+---------------------+
|started_at                      |ended_at               

Total rows: 8432882
Columns: ['started_at', 'ended_at', 'duration', 'start_station_id', 'start_station_name', 'start_station_description', 'start_station_latitude', 'start_station_longitude', 'end_station_id', 'end_station_name', 'end_station_description', 'end_station_latitude', 'end_station_longitude']


- We have a total of 8 428 152 biketrips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `endend_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We also have the added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as string data types. This is because PySpark reads CSV columns as strings by default. However, Several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> integer(maybe double?)
- Coorinates: `latitude`and `longitude`-> double

Maybe:
- StationIDs: `start_station_id` and `stop_station_id` -> keep as string og convert to integer

From the table above we can see the ID is stored as a numeric value, but we would argue to keep them as a string as they are identifiers rather than numeric measurements. This also provides a greater flexibility if future datasets contains letters(e.g. ST123), without requiring changes to the data type. 


### Investigating missing values

In [ ]:
trips_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips.columns
]).show()

+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+
|started_at|ended_at|duration|start_station_id|start_station_name|start_station_description|start_station_latitude|start_station_longitude|end_station_id|end_station_name|end_station_description|end_station_latitude|end_station_longitude|
+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+
|         0|       0|       0|               0|                 0|                    11018|                     0|                      0|             0|               0|                   6962|                   0|                    0|
+----------+--------+--------+--------------

#### Missing values
The `start_station_description` column contains 11 000 missing values, while `end_station_description` contains 6950 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysing we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station ID and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to are final analytical dataset. 

### Investigating duplicates

In [ ]:
print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_raw.distinct().count())

Total rows: 8432882


26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/02 11:42:17 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate rows: 25


Out of the 8 428 152 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We can not be certain that these are dublicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 collumns, including timestamp with microseconds precision. It is highly unlikely that separate trips would share all these values.

We consider therefore these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contain over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset was significanlty smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [ ]:
trips_silver = trips_raw.dropDuplicates()

### Investigatin values and distributions

#### First Duration
Lets convert the duration to double 

In [ ]:
trips_silver = trips_silver.withColumn("duration", col("duration").cast("double"))

In [ ]:

trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

+-------+----------+
|summary|duration_s|
+-------+----------+
|    min|       0.0|
|    25%|     341.0|
|    50%|     524.0|
|    75%|     806.0|
|    max|   79790.0|
+-------+----------+



+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1273|            0|
+------------+---------+-------------+



#### Durations to investigate ferther
- Is it possible that a trip last around 22 hours?
- 2 trips has a negative durations

In [ ]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

### Extremely Short Trips
29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need futher investigation.

Possible solution: 
- Find the mean trip duration from one statiotn to the other and compare it to minimum duration record.
- Flag trips under a spesific timeframe. 


In [ ]:
station_durations = (
    trips_silver
    .groupBy("start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

### What does the table above tell us?
Although the mean duration does not accually tell us the minimum possible duration between two stations, it provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundres and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only one second. This strongly suggest that some values are incorrect or unrealistic. 

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's atleast flag trips with a duration smaller than one second. 

In [ ]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

### Trips with a long duration

In [ ]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration_s") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration_s"
).orderBy(desc("duration")).show(15, truncate=False)

Number of trips longer than 12 hours:  15


+-----------------------+-----------------------+----------+
|started_at             |ended_at               |duration_s|
+-----------------------+-----------------------+----------+
|2026-06-19 13:38:13.233|2026-06-20 11:48:03.076|79790.0   |
|2025-10-17 14:35:20.232|2025-10-18 11:15:52.091|74431.0   |
|2022-04-16 12:47:44.363|2022-04-17 07:37:10.178|67765.0   |
|2023-04-07 13:40:11.612|2023-04-08 08:18:37.147|67105.0   |
|2023-05-31 15:31:03.704|2023-06-01 08:37:01.096|61557.0   |
|2026-05-04 18:28:16.356|2026-05-05 11:14:23.84 |60367.0   |
|2022-11-01 15:19:05.897|2022-11-02 07:26:44.435|58058.0   |
|2022-11-24 14:20:12.725|2022-11-25 05:44:40.784|55468.0   |
|2026-08-06 20:38:28.625|2026-08-07 11:37:01.191|53913.0   |
|2022-05-20 16:49:31.692|2022-05-21 06:44:45.835|50114.0   |
|2022-10-10 19:37:20.877|2022-10-11 09:10:35.12 |48794.0   |
|2022-05-15 15:36:45.755|2022-05-16 04:41:07.477|47061.0   |
|2026-10-19 16:33:30.177|2026-10-20 05:36:35.162|46985.0   |
|2022-11-15 15:53:39.71 

Could a city bike realistically be used for appocimately 22 hours?

Accoring to Oslo Bysykkel, a fully charged electric bike has an estimate range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explainations for the long duration
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [ ]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

#### Does duration correspond to the start and end time?

In [ ]:
trips_silver.select(
    "started_at",
    "ended_at",
    "duration",
    # does started_at and ended_at match the duration?
    (to_timestamp(col("ended_at")).cast("long") - to_timestamp(col("started_at")).cast("long")).alias("duration_check")
    ).show(5, truncate=False)

+-----------------------+-----------------------+--------+--------------+
|started_at             |ended_at               |duration|duration_check|
+-----------------------+-----------------------+--------+--------------+
|2022-08-01 03:08:45.036|2022-08-01 03:11:38.571|173     |173           |
|2022-08-01 03:09:05.419|2022-08-01 03:18:15.885|550     |550           |
|2022-08-01 03:22:25.178|2022-08-01 03:30:57.051|511     |512           |
|2022-08-01 03:29:20.355|2022-08-01 03:38:43.156|562     |563           |
|2022-08-01 03:29:25.883|2022-08-01 03:37:00.311|454     |455           |
+-----------------------+-----------------------+--------+--------------+
only showing top 5 rows



Since we used the `.cast("long")`we lose miliseconds, which can explain the reason for the 1 second difference. 

for example:
- 511 and 512
- 562 and 563
- 454 and 455

Let's investigate the duration a bit further.

In [ ]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros(to_timestamp("ended_at"))
         - unix_micros(to_timestamp("started_at"))) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

+-------+-------------------+
|summary|      duration_diff|
+-------+-------------------+
|  count|            8428152|
|   mean|0.45660269961908617|
| stddev| 0.2853735155457395|
|    min|                0.0|
|    25%|0.21299999999996544|
|    50%|0.42699999999990723|
|    75%| 0.6979999999999791|
|    max| 0.9990000000000236|
+-------+-------------------+



Since the maximum difference are below one second, we consider the recorded durations consistent with the timestamps. 

### Investigating the timestamp format

In [ ]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

NameError: name 'trips' is not defined

#### Converting the string to UTC timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [ ]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_at             |started_ts             |ended_at               |ended_ts               |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 03:08:45.036|2022-08-01 03:08:45.036|2022-08-01 03:11:38.571|2022-08-01 03:11:38.571|
|2022-08-01 03:09:05.419|2022-08-01 03:09:05.419|2022-08-01 03:18:15.885|2022-08-01 03:18:15.885|
|2022-08-01 03:22:25.178|2022-08-01 03:22:25.178|2022-08-01 03:30:57.051|2022-08-01 03:30:57.051|
|2022-08-01 03:29:20.355|2022-08-01 03:29:20.355|2022-08-01 03:38:43.156|2022-08-01 03:38:43.156|
|2022-08-01 03:29:25.883|2022-08-01 03:29:25.883|2022-08-01 03:37:00.311|2022-08-01 03:37:00.311|
|2022-08-01 03:30:15.563|2022-08-01 03:30:15.563|2022-08-01 03:45:27.332|2022-08-01 03:45:27.332|
|2022-08-01 03:36:52.062|2022-08-01 03:36:52.062|2022-08-01 03:50:05.634|2022-08-01 03:50:05.634|
|2022-08-01 03:38:34

In [ ]:
# Did the conversion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [ ]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp(
            to_timestamp(col("started_ts")),
            "Europe/Oslo"
        )
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp(
            to_timestamp(col("ended_ts")),
            "Europe/Oslo"
        )
    )
)

trips.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_ts             |started_local          |ended_ts               |ended_local            |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 03:08:45.036|2022-08-01 05:08:45.036|2022-08-01 03:11:38.571|2022-08-01 05:11:38.571|
|2022-08-01 03:09:05.419|2022-08-01 05:09:05.419|2022-08-01 03:18:15.885|2022-08-01 05:18:15.885|
|2022-08-01 03:22:25.178|2022-08-01 05:22:25.178|2022-08-01 03:30:57.051|2022-08-01 05:30:57.051|
|2022-08-01 03:29:20.355|2022-08-01 05:29:20.355|2022-08-01 03:38:43.156|2022-08-01 05:38:43.156|
|2022-08-01 03:29:25.883|2022-08-01 05:29:25.883|2022-08-01 03:37:00.311|2022-08-01 05:37:00.311|
|2022-08-01 03:30:15.563|2022-08-01 05:30:15.563|2022-08-01 03:45:27.332|2022-08-01 05:45:27.332|
|2022-08-01 03:36:52.062|2022-08-01 05:36:52.062|2022-08-01 03:50:05.634|2022-08-01 05:50:05.634|
|2022-08-01 03:38:34

### Number of trips divided by the different cities

In [ ]:
trips_silver = trips_silver.withColumn(
    "src", input_file_name()
)

trips_silver = trips_silver.withColumn(
    "city", element_at(split(col("src"), "/"), -2)
)

trips_silver = trips_silver.withColumn(
    "year_month",
    split(element_at(split(col("src"), "/"), -1), r"\.").getItem(0)
)

trips_silver.select("city", "year_month").show(5)

trips_silver.groupBy("city").count().show()

+---------+-------+
|     city|  count|
+---------+-------+
|     oslo|5718138|
|   bergen|1904236|
|trondheim| 805778|
+---------+-------+

